In [ ]:
import requests
import csv
from time import sleep
from dotenv import load_dotenv
import os
from datetime import datetime, timedelta



load_dotenv(dotenv_path="../.env")
API_KEY = os.getenv("API_KEY")



counties_names = [
    'Bistrita-Nasaud', 'Maramures', 'Satu Mare', 'Salaj', 'Bihor', 'Cluj', 'Alba',
    'Bucuresti', 'Suceava', 'Mures', 'Sibiu', 'Brasov', 'Covasna', 'Harghita',
    'Neamt', 'Bacau', 'Arad', 'Hunedoara', 'Botosani', 'Iasi', 'Vaslui', 'Vrancea',
    'Galati', 'Gorj', 'Valcea', 'Arges', 'Dambovita', 'Prahova', 'Buzau', 'Braila',
    'Ialomita', 'Timis', 'Caras-Severin', 'Mehedinti', 'Dolj', 'Olt', 'Teleorman',
    'Giurgiu', 'Ilfov', 'Calarasi', 'Constanta', 'Tulcea'
]

cities = [
    'Bistrita', 'Baia Mare', 'Satu Mare', 'Zalau', 'Oradea', 'Cluj-Napoca', 'Alba Iulia',
    'Bucuresti', 'Suceava', 'Targu Mures', 'Sibiu', 'Brasov', 'Sfantu Gheorghe', 'Miercurea Ciuc',
    'Piatra Neamt', 'Bacau', 'Arad', 'Hunedoara', 'Botosani', 'Iasi', 'Vaslui', 'Focsani',
    'Galati', 'Targu Jiu', 'Râmnicu Valcea', 'Pitesti', 'Targoviste', 'Ploiesti', 'Buzau', 'Braila',
    'Slobozia', 'Timisoara', 'Resita', 'Drobeta-Turnu Severin', 'Craiova', 'Slatina', 'Alexandria',
    'Giurgiu', 'Otopeni', 'Calarasi', 'Constanta', 'Tulcea'
]

counties_abbrev = [
    'BN', 'MM', 'SM', 'SJ', 'BH', 'CJ', 'AB',
    'B',  'SV', 'MS', 'SB', 'BV', 'CV', 'HR',
    'NT', 'BC', 'AR', 'HD', 'BT', 'IS', 'VS', 'VN',
    'GL', 'GJ', 'VL', 'AG', 'DB', 'PH', 'BZ', 'BR',
    'IL', 'TM', 'CS', 'MH', 'DJ', 'OT', 'TR',
    'GR', 'IF', 'CL', 'CT', 'TL'
]

counties_id = list(range(0, len(counties_names)))




headers = [
        "county_id", "county", "city", "county_abbrev", "data_type", "time_epoch", "time", "temp_c", "temp_f",
        "is_day", "condition_text", "condition_icon", "wind_mph",
        "wind_kph", "wind_degree", "wind_dir", "pressure_mb",
        "pressure_in", "precip_mm", "precip_in", "snow_cm",
        "humidity", "cloud", "feelslike_c", "feelslike_f",
        "windchill_c", "windchill_f", "heatindex_c", "heatindex_f",
        "dewpoint_c", "dewpoint_f", "will_it_rain", "chance_of_rain",
        "will_it_snow", "chance_of_snow", "vis_km", "vis_miles",
        "gust_mph", "gust_kph", "uv", "short_rad",
        "diff_rad", "dni", "gti"
    ]

data_table = []
data_table.append(headers)


today = datetime.now()
start_date = (today - timedelta(6)).strftime("%Y-%m-%d")
end_date = (today - timedelta(1)).strftime("%Y-%m-%d")

urls = {"history": f"https://api.weatherapi.com/v1/history.json?q=LOCATION&dt={start_date}&end_dt={end_date}&lang=ro&key={API_KEY}",
        "forecast": f"https://api.weatherapi.com/v1/forecast.json?q=LOCATION&days=3&lang=ro&key={API_KEY}"}


for index in range(0, len(cities)):
    for data_type in urls:
        url = urls[data_type].replace("LOCATION", f"{cities[index]},{counties_names[index]},Romania")
        header = {"accept": "application/json"}

        response = requests.get(url, headers = header)
        status_code = response.status_code

        print(f"Response code: {status_code}")

        if status_code == 200:
            data = response.json()
            print(f"Collecting {data_type} data from: {data["location"]["name"]} / {data["location"]["region"]} / {data["location"]["country"]}")
            forecast_data = data["forecast"]["forecastday"]


            for day_weather in forecast_data:
                print(f"Date: {day_weather["date"]}")
                for hour_weather in day_weather["hour"]:
                    row = []
                    row.append(counties_id[index])
                    row.append(counties_names[index])
                    row.append(cities[index])
                    row.append(counties_abbrev[index])
                    row.append(data_type)
                    for info in hour_weather:
                        if info == "condition":
                            row.append(hour_weather["condition"]["text"])
                            row.append(hour_weather["condition"]["icon"])
                        else:
                            row.append(hour_weather[info])

                    data_table.append(row)
        
        sleep(1)
    
    print("-------------------------------")



file_path = r"../data/raw/WeatherData_raw.csv"
print(f"Saving file: {file_path}")
file = open(file_path, "w", newline = "", encoding = "UTF-8-sig")
writer = csv.writer(file, delimiter = "|")
writer.writerows(data_table)
file.close()
                